# Auth & Rate Limits

Every DataCore call is authenticated with an API key and counted against your tier's quota. Understanding how to handle 429s gracefully is the difference between a notebook that finishes and one that crashes 80% through a backtest.

**Why this matters**: rate limits are the most common production failure mode. A backoff-aware client recovers automatically; a naive one loses state and forces a full restart.

## Setting your API key

DataCore's `Client` reads `DATACORE_API_KEY` from the environment. You can also pass it explicitly — useful in notebooks where you want to switch between accounts.

In [ ]:
import os

# Option 1: environment variable (recommended)
# export DATACORE_API_KEY=dc_live_xxx in your shell

# Option 2: set inline (avoid committing this!)
os.environ['DATACORE_API_KEY'] = os.environ.get('DATACORE_API_KEY', 'dc_demo_replace_me')

from datacore import Client

dc = Client()  # picks up env var automatically
print('Authenticated as:', dc.whoami())

## Free vs paid tiers

| Tier      | Requests / min | Datasets         | Historical depth | Price (VND/mo) |
|-----------|----------------|------------------|------------------|----------------|
| Free      | 30             | EOD equity only  | 2 years          | 0              |
| Researcher| 300            | + fundamentals   | 10 years         | 490,000        |
| Pro       | 3,000          | + alt-data, intraday | full history | 4,900,000      |
| Enterprise| custom         | everything       | full history     | contact sales  |

Check your current tier and remaining quota at any time:

In [ ]:
quota = dc.quota()
print(f"Tier: {quota['tier']}")
print(f"Remaining this minute: {quota['remaining']} / {quota['limit']}")
print(f"Resets in: {quota['reset_seconds']}s")

## Handling `RateLimitError`

When you exceed your per-minute quota, DataCore raises a `RateLimitError` with a `retry_after` attribute. The simplest correct pattern is to sleep and retry.

In [ ]:
import time
from datacore.exceptions import RateLimitError

def safe_pull(dataset_id, **kwargs):
    try:
        return dc.dataset(dataset_id).to_pandas(**kwargs)
    except RateLimitError as e:
        print(f'Rate limited. Sleeping {e.retry_after}s…')
        time.sleep(e.retry_after)
        return dc.dataset(dataset_id).to_pandas(**kwargs)

df = safe_pull('equity.vn30.daily', start='2024-01-01', end='2024-01-31')
df.head()

## Exponential backoff for long-running jobs

For batch jobs that pull thousands of tickers, a single retry isn't enough — you want exponential backoff with jitter so concurrent workers don't thunder back together.

In [ ]:
import random
from datacore.exceptions import RateLimitError, ServerError

def pull_with_backoff(dataset_id, max_attempts=6, base=1.0, cap=60.0, **kwargs):
    for attempt in range(max_attempts):
        try:
            return dc.dataset(dataset_id).to_pandas(**kwargs)
        except (RateLimitError, ServerError) as e:
            if attempt == max_attempts - 1:
                raise
            # exponential backoff with full jitter
            sleep_s = min(cap, base * (2 ** attempt)) * random.random()
            print(f'Attempt {attempt+1} failed: {type(e).__name__}. Sleeping {sleep_s:.1f}s')
            time.sleep(sleep_s)

# Pull a year of VN30 with resilience
df = pull_with_backoff('equity.vn30.daily', start='2024-01-01', end='2024-12-31')
print(f'Pulled {len(df):,} rows')

## Checking quota before a big pull

Before kicking off a 500-ticker scrape, peek at the quota and bail early if you'd run out:

In [ ]:
tickers = ['VNM', 'VHM', 'VIC', 'FPT', 'HPG', 'MWG', 'VCB', 'TCB', 'MSN', 'GAS']

q = dc.quota()
if q['remaining'] < len(tickers):
    wait = q['reset_seconds'] + 1
    print(f'Only {q["remaining"]} requests left, need {len(tickers)}. Waiting {wait}s.')
    time.sleep(wait)

frames = {}
for t in tickers:
    frames[t] = pull_with_backoff('equity.daily', ticker=t, start='2024-01-01')

print(f'Pulled {len(frames)} tickers, quota now: {dc.quota()["remaining"]} requests left')

## Next steps

- `03-caching-and-performance.ipynb` — stop hitting the API for repeat pulls
- `02-equity-research/01-vn30-fundamentals.ipynb` — use these patterns in a real screener
- [Tier comparison & pricing](https://datacore.vn/pricing)